# 2. Training the MLC-AETrains the multi-task autoencoder (reconstruction + G0/non-G0 classification) on each dataset and saves the weights. The released weights in `weights/` were produced by exactly this step.**Inputs:** normalized matrices, `results/common_genes.txt`**Outputs:** `weights/<dataset>_mlcae.weights.h5`, `weights/<dataset>_mlcae.json`**Approximate runtime:** about 2-8 minutes per dataset on a laptop GPU (NVIDIA RTX 2060); longer on CPUPaths are set in `scripts/common.py` (`H5_DIR`, `RAW_DIR`, `OUT`). Edit them once before running any notebook.

In [ ]:
import sys, jsonfrom pathlib import Pathsys.path.insert(0, str(Path.cwd().parent / 'scripts'))import common as Cprint('datasets:', list(C.DATASETS))print('results directory:', C.OUT)

Run the training script for one dataset (or omit the argument to train all six):

In [ ]:
%run ../train_weights.py PC3_high

### Load released weights and score the held-out cells

In [ ]:
import numpy as np, jsonfrom sklearn.model_selection import train_test_splitfrom sklearn.metrics import roc_auc_scorefrom ribo_ablation import common_genes, load_densesys.path.insert(0, str(Path.cwd().parent))from train_weights import buildname = 'PC3_high'genes = common_genes(); X, y = load_dense(name, genes)xtr, xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=4, stratify=y)model = build(X.shape[1]); model.load_weights(f'../weights/{name}_mlcae.weights.h5')p = model.predict(xte, batch_size=2048, verbose=0)[1][:, 1]print('held-out AUC:', round(roc_auc_score(yte, p), 4))print('recorded at release:', json.load(open(f'../weights/{name}_mlcae.json'))['test_auc'])